In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import xarray as xr
from context_flux_no.simulations.pde import Burgers1D
from context_flux_no.waveforms.fourier import TruncatedFourier1D
from context_flux_no.waveforms.grf import (
    GaussianCov,
    GaussianRandomField1D,
)
from tqdm import tqdm


jax.config.update("jax_enable_x64", True)

In [ ]:
x = jnp.linspace(0, 1, 256)
u0 = GaussianRandomField1D(GaussianCov(0.1)).sample(x, jax.random.key(1))

fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.plot(x, u0)

In [ ]:
u0 = TruncatedFourier1D.with_uniform_rand_coeffs(num_modes=4)
x = jnp.linspace(0, 1, 100)
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.plot(x, u0(x))

In [ ]:
grf = GaussianRandomField1D(GaussianCov(0.1))
u0_func = lambda x_: grf.sample(x_, key=jax.random.key(0))
burgers = Burgers1D()
# u0_func = TruncatedFourier1D.with_uniform_rand_coeffs(num_modes=4)

sol = burgers.solve_pyclaw(u0_func, (0.0, 1.0), 256, (0.0, 0.2), 500, "periodic")

In [ ]:
sol

In [ ]:
t_vals = (0, 124, 249, 374, 499)
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharex=True)
u = sol["values"]
for j, t_i in enumerate(t_vals):
    axes[j].plot(u["x"], u.isel(ic=0, t=t_i, dim=0))

In [ ]:
NUM_SAMPLES = 50
SEED = 0


burgers = Burgers1D()
grf = GaussianRandomField1D(GaussianCov(0.1))

keys = jax.random.split(jax.random.key(SEED), NUM_SAMPLES)
solutions = []
for key in tqdm(keys):
    u0_func = lambda x_: grf.sample(x_, key=key)
    sol = burgers.solve_pyclaw(
        u0_func, (0.0, 1.0), 256, (0.0, 0.2), 500, "periodic", verbose=False
    )
    solutions.append(sol)

In [ ]:
dataset = xr.concat(solutions, "pde")

In [ ]:
dataset

In [ ]:
savedir = Path("../../data/")
savedir.mkdir(parents=True, exist_ok=True)
dataset.to_netcdf(savedir / "burgers_train.hdf5")

## Step function

In [ ]:
step = PeriodicRandomStepFunction1D()
u0_step = step.sample(x, jax.random.key(2))